# HL Portfolio & PnL Dashboard

Tracks account balances, open perp positions, and realized/unrealized PnL including trading fees and signed perp funding from HL data cached in CSV. Calculates portfolio performance and risk metrics (including historical and bootstrap VaR and historical volatility), attributes PnL by asset, and compares account performance with BTC through charts and summary statistics.


In [ ]:
from desk.reloading import reload_desk
reload_desk()  # pick up code changes in every desk module without restarting the kernel
import desk.hl.account_analytics as implementation

# Enter your wallet address here to fetch data for that specific wallet
WALLET_ADDRESS = "0x...07f2"
# WALLET_ADDRESS = "0x...c248"

ATR_PERIOD = 14
ATR_STOP_MULTIPLE = 1.25
MARGIN_RESERVE_PCT = 20.0
AGGREGATE_RISK_PCT = 20.0

analytics = implementation.HLAnalytics(WALLET_ADDRESS)
analytics.position_atr_period = ATR_PERIOD
analytics.position_atr_stop_multiple = ATR_STOP_MULTIPLE
analytics.margin_reserve_pct = MARGIN_RESERVE_PCT
analytics.aggregate_risk_pct = AGGREGATE_RISK_PCT
_ = analytics.fetch_data()

## Highlights

What went well and what needs attention since this notebook last ran, over the last 24 hours, and right now. PnL includes open positions and excludes deposits and withdrawals. Each run saves a baseline per wallet in `data/hl_highlights_state_<wallet>.json`.

In [ ]:
analytics.show_highlights()

## Account / PnL

Interactive account and risk summary.

**Account value** is calculated live from current balances and, on accounts that are not unified, perp equity. It is not the open-position notional. 24H rows are rolling windows ending now and exclude deposits and withdrawals. Daily, weekly, and monthly returns, drawdown, Sharpe, alpha, and beta use a time-weighted index from HL history that excludes deposits and withdrawals, saved in `data/hl_portfolio_history_<wallet>.csv`; daily returns need history points on consecutive US Eastern days. Alpha, beta, and Sharpe use completed US Eastern days only, with BTC priced at the same local midnight, and show 90% bootstrap ranges. **Current book BTC beta** is the open positions' signed notional times each asset's 60-day beta to BTC, per dollar of account value. A warning appears when the latest history value differs from the live value by more than 2%.

In [ ]:
analytics.show_account_pnl()

In [ ]:
analytics.show_net_pnl()

## Account charts

Interactive Plotly charts show account value, daily realized and unrealized PnL, cumulative PnL, and weekly return distribution.

In [ ]:
analytics.show_account_charts()

## Account vs BTC

Shows a time-weighted account index against BTC, so deposits, withdrawals, and transfers do not move it.

In [ ]:
analytics.show_account_vs_btc()

## Traded volume vs realized PnL chart

Daily fill volume compared with realized PnL by market type.

In [ ]:
analytics.show_volume_chart()

## Open positions

Position tables, net exposure, funding by asset and side, and an account summary. The position tables include margin type, funding and fees since open as dollar amounts, fees as a percentage of position value, and position value as a percentage of account value. Fee ratios are available for complete USDC-denominated fee history. Reserve-adjusted available margin is margin collateral (USDC on unified accounts) less `MARGIN_RESERVE_PCT`. Aggregate risk used and room apply `AGGREGATE_RISK_PCT` to the same collateral, using `ATR_PERIOD` and `ATR_STOP_MULTIPLE` stops plus an adverse five-day funding reserve.

In [ ]:
analytics.show_open_positions()

In [ ]:
_ = analytics.show_btc_exposure()

## Net funding APR over time

Applies the open-positions `Net Funding APR (%) (simple, covered)` formula to HL funding history: net funding cash flow divided by gross notional, annualized. The history records actual funding on positions held at the time; the summary table shows the current rate on current positions, so the two differ. Daily values use US Eastern days, and the 7-day line is notional-weighted. Positive means funding received.

In [ ]:
_ = analytics.show_funding_apr_chart()

## Realized PnL statistics

Interactive tables report fee-net spot/perp results with signed perp funding, wins, losses, and totals. The perp output also includes liquidation count, share of closes, and liquidation PnL. Daily buckets use US Eastern time; 24H, 7D, and 30D windows are elapsed-time periods.


In [ ]:
analytics.show_realized_pnl()

## Traded volume vs PnL

Interactive table compares turnover efficiency and correlations by market type. Realized PnL / Volume (%) is realized fill PnL divided by one-sided traded notional; funding is excluded.

In [ ]:
analytics.show_volume_summary()

## Execution quality

Last 30 days of perp fills split into opening and closing, maker and taker, with the fees paid. Stop-losses must stay taker, so closing taker fills are only partly avoidable.

In [ ]:
analytics.show_execution_quality()

## PnL attribution

Three tables, crypto perps, TradFi perps, then spot, break realized PnL out by asset; the perp tables add open-position uPnL and total PnL. The Closes column shows distinct closes per asset, counted from available fills. Each percentage column is the signed contribution as a share of that column's gross absolute PnL within its table, so gains are positive, losses are negative, and the colors match the dollar values. Percentages can sum to less than 100% when gains and losses offset. The spot table's Other row holds vaults, transferred tokens, and open spot positions: HL's non-perp PnL less spot fill PnL.

In [ ]:
analytics.show_pnl_attribution()